### resolve csv.gz.icloud issue after downloading files from google drive using gdown
is a placeholder created by macOS iCloud Drive optimization. It means:  
- The actual file is in iCloud, not on your disk.  
- Python (or any app outside Finder) cannot read it until it’s fully downloaded.  

In [1]:
import os
print('-'*40, 'hosp', '-'*40)
path = "/Users/ginger/Downloads/DSCI531_Project/data/mimic4/hosp"
for i in os.listdir(path):
    print(i)
print('-'*40, 'icu', '-'*40)
path = "/Users/ginger/Downloads/DSCI531_Project/data/mimic4/icu"
for i in os.listdir(path):
    print(i)

---------------------------------------- hosp ----------------------------------------
d_hcpcs.csv.gz
.poe_detail.csv.gz.icloud
patients.csv.gz
.DS_Store
.hcpcsevents.csv.gz.icloud
diagnoses_icd.csv.gz
emar_detail.csv.gz
.prescriptions.csv.gz.icloud
drgcodes.csv.gz
d_icd_diagnoses.csv.gz
d_labitems.csv.gz
transfers.csv.gz
admissions.csv.gz
.microbiologyevents.csv.gz.icloud
labevents.csv.gz
.poe.csv.gz.icloud
procedures_icd.csv.gz
services.csv.gz
d_icd_procedures.csv.gz
.pharmacy.csv.gz.icloud
.omr.csv.gz.icloud
emar.csv.gz
---------------------------------------- icu ----------------------------------------
.outputevents.csv.gz.icloud
.DS_Store
.datetimeevents.csv.gz.icloud
.ingredientevents.csv.gz.icloud
d_items.csv.gz
chartevents.csv.gz
.procedureevents.csv.gz.icloud
icustays.csv.gz
outputevents.csv.gz


In [ ]:
import pandas as pd

# Core demographic and admission data
patients = pd.read_csv("data/mimic4/hosp/patients.csv.gz")
admissions = pd.read_csv("data/mimic4/hosp/admissions.csv.gz")

# ICU stays
icustays = pd.read_csv("data/mimic4/icu/icustays.csv.gz")

# Diagnoses (optional but useful)
diagnoses = pd.read_csv("data/mimic4/hosp/diagnoses_icd.csv.gz")

# Charted ICU data (vitals)
cols = ['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum']
chartevents = pd.read_csv("data/mimic4/icu/chartevents.csv.gz", usecols=cols)

# Lab tests
cols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum']
labevents = pd.read_csv("data/mimic4/hosp/labevents.csv.gz", usecols=cols)

# Dictionaries for interpretation
d_items = pd.read_csv("data/mimic4/icu/d_items.csv.gz")
d_labitems = pd.read_csv("data/mimic4/hosp/d_labitems.csv.gz")


## Preprocessing

In [24]:
patients.head()  

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [25]:
print(f'''patients:
{patients.shape}
{'-'*100}
{patients.dtypes}
{'-'*100}
{patients.count()}
{'-'*100}
{patients.isna().sum()}
{'-'*100}
{patients.describe()}
{'-'*100}
''')

patients:
(364627, 6)
----------------------------------------------------------------------------------------------------
subject_id            int64
gender               object
anchor_age            int64
anchor_year           int64
anchor_year_group    object
dod                  object
dtype: object
----------------------------------------------------------------------------------------------------
subject_id           364627
gender               364627
anchor_age           364627
anchor_year          364627
anchor_year_group    364627
dod                   38301
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id                0
gender                    0
anchor_age                0
anchor_year               0
anchor_year_group         0
dod                  326326
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id     anchor

In [35]:
# Drop synthetic time columns
patients_clean = patients.copy()
patients_clean = patients_clean.drop(columns=['anchor_year', 'anchor_year_group'])

# Convert gender to binary
patients_clean['gender'] = patients_clean['gender'].map({'M': 1, 'F': 0})

# Convert dod to datetime
patients_clean['dod'] = pd.to_datetime(patients_clean['dod'])

# dod will have many missing values — that’s expected (most patients survived)
patients_clean.info()
patients_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   subject_id  364627 non-null  int64         
 1   gender      364627 non-null  int64         
 2   anchor_age  364627 non-null  int64         
 3   dod         38301 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(3)
memory usage: 11.1 MB


,subject_id,gender,anchor_age,dod
0,10000032,0,52,2180-09-09
1,10000048,0,23,NaT
2,10000058,0,33,NaT
3,10000068,0,19,NaT
4,10000084,1,72,2161-02-13


In [31]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [37]:
print(f'''admissions:
{admissions.shape}
{'-'*100}
{admissions.dtypes}
{'-'*100}
{admissions.count()}
{'-'*100}
{admissions.isna().sum()}
{'-'*100}
{admissions.describe()}
{'-'*100}
''')

admissions:
(546028, 16)
----------------------------------------------------------------------------------------------------
subject_id               int64
hadm_id                  int64
admittime               object
dischtime               object
deathtime               object
admission_type          object
admit_provider_id       object
admission_location      object
discharge_location      object
insurance               object
language                object
marital_status          object
race                    object
edregtime               object
edouttime               object
hospital_expire_flag     int64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id              546028
hadm_id                 546028
admittime               546028
dischtime               546028
deathtime                11790
admission_type          546028
admit_provider_id       546024
admission_location      546027
discharge_locat

In [ ]:
cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'insurance', 'language', 'marital_status', 'race',
             'hospital_expire_flag', 'admission_type']

admissions_clean = admissions[cols].copy()

# Convert datetime columns
admissions_clean['admittime'] = pd.to_datetime(admissions_clean['admittime'])
admissions_clean['dischtime'] = pd.to_datetime(admissions_clean['dischtime'])

# Fill missing values for sensitive attributes
'''“This particular NaN value has special meaning — it tells us that the information was unknown or missing at the time. 
Instead of dropping or guessing it, we treat it as its own category by converting it into a label like 'UNKNOWN', so the model can process it.”'''

admissions_clean[['insurance', 'language', 'marital_status']] = admissions_clean[['insurance', 'language', 'marital_status']].fillna('UNKNOWN')

# Preview cleaned data
admissions_clean.info()
admissions_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int64         
 1   hadm_id               546028 non-null  int64         
 2   admittime             546028 non-null  datetime64[ns]
 3   dischtime             546028 non-null  datetime64[ns]
 4   insurance             546028 non-null  object        
 5   language              546028 non-null  object        
 6   marital_status        546028 non-null  object        
 7   race                  546028 non-null  object        
 8   hospital_expire_flag  546028 non-null  int64         
 9   admission_type        546028 non-null  object        
dtypes: datetime64[ns](2), int64(3), object(5)
memory usage: 41.7+ MB


,subject_id,hadm_id,admittime,dischtime,insurance,language,marital_status,race,hospital_expire_flag,admission_type
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,Medicaid,English,WIDOWED,WHITE,0,URGENT
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,Medicaid,English,WIDOWED,WHITE,0,EW EMER.
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,Medicaid,English,WIDOWED,WHITE,0,EW EMER.
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,Medicaid,English,WIDOWED,WHITE,0,EW EMER.
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,UNKNOWN,English,SINGLE,WHITE,0,EU OBSERVATION


In [7]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [38]:
print(f'''icustays:
{icustays.shape}
{'-'*100}
{icustays.dtypes}
{'-'*100}
{icustays.count()}
{'-'*100}
{icustays.isna().sum()}
{'-'*100}
{icustays.describe()}
{'-'*100}
''')

icustays:
(94458, 8)
----------------------------------------------------------------------------------------------------
subject_id          int64
hadm_id             int64
stay_id             int64
first_careunit     object
last_careunit      object
intime             object
outtime            object
los               float64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id        94458
hadm_id           94458
stay_id           94458
first_careunit    94458
last_careunit     94458
intime            94458
outtime           94444
los               94444
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id         0
hadm_id            0
stay_id            0
first_careunit     0
last_careunit      0
intime             0
outtime           14
los               14
dtype: int64
--------------------------------------------------------------------

In [39]:
'''Don't use los for ICU mortality prediction because it creates a leakage risk.
If someone dies early, their los will be very short → model might learn to predict death from low los directly.
You want to predict it based on vitals, labs, and demographics available early, so use los for descriptive statistics and filtering out noise'''


# You can’t define ICU mortality without outtime so drop those missing rows
# Drops entire rows where the column outtime is NaN
icustays_clean = icustays.copy()
icustays_clean = icustays_clean.dropna(subset=['outtime'])

# Convert intime and outtime to datetime
icustays_clean['intime'] = pd.to_datetime(icustays_clean['intime'])
icustays_clean['outtime'] = pd.to_datetime(icustays_clean['outtime'])

cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los', 'first_careunit']
icustays_clean = icustays_clean[cols]

# Preview cleaned data
icustays_clean.info()
icustays_clean.head()

<class 'pandas.core.frame.DataFrame'>
Index: 94444 entries, 0 to 94457
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_id      94444 non-null  int64         
 1   hadm_id         94444 non-null  int64         
 2   stay_id         94444 non-null  int64         
 3   intime          94444 non-null  datetime64[ns]
 4   outtime         94444 non-null  datetime64[ns]
 5   los             94444 non-null  float64       
 6   first_careunit  94444 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(3), object(1)
memory usage: 5.8+ MB


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,Medical Intensive Care Unit (MICU)
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,Medical Intensive Care Unit (MICU)
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,Medical Intensive Care Unit (MICU)
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,Surgical Intensive Care Unit (SICU)
4,10001217,27703517,34592300,2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,Surgical Intensive Care Unit (SICU)


In [9]:
diagnoses.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [ ]:
print(f'''diagnoses:
{diagnoses.shape}
{'-'*100}
{diagnoses.dtypes}
{'-'*100}
{diagnoses.count()}
{'-'*100}
{diagnoses.isna().sum()}
{'-'*100}
{diagnoses.describe()}
{'-'*100}
''')

diagnoses:
(6364488, 5)
----------------------------------------------------------------------------------------------------
subject_id     6364488
hadm_id        6364488
seq_num        6364488
icd_code       6364488
icd_version    6364488
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id     0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id       seq_num   icd_version
count  6.364488e+06  6.364488e+06  6.364488e+06  6.364488e+06
mean   1.500237e+07  2.500059e+07  8.825529e+00  9.542973e+00
std    2.878400e+06  2.889094e+06  6.858762e+00  4.981499e-01
min    1.000003e+07  2.000002e+07  1.000000e+00  9.000000e+00
25%    1.251032e+07  2.249520e+07  4.000000e+00  9.000000e+00
50%    1.500206e+07  2.500402e+07  7.000000e+00  1.000000e+01
75%    1.7501

In [11]:
chartevents.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
0,10000032,29079034,39553978,2180-07-23 12:36:00,226512,39.4
1,10000032,29079034,39553978,2180-07-23 12:36:00,226707,60.0
2,10000032,29079034,39553978,2180-07-23 12:36:00,226730,152.0
3,10000032,29079034,39553978,2180-07-23 14:00:00,220048,NaN
4,10000032,29079034,39553978,2180-07-23 14:00:00,224642,NaN


In [ ]:
print(f'''chartevents:
{chartevents.shape}
{'-'*100}
{chartevents.dtypes}
{'-'*100}
{chartevents.count()}
{'-'*100}
{chartevents.isna().sum()}
{'-'*100}
{chartevents.describe()}
{'-'*100}
''')

chartevents:
(432997491, 6)
----------------------------------------------------------------------------------------------------
subject_id    432997491
hadm_id       432997491
stay_id       432997491
charttime     432997491
itemid        432997491
valuenum      169211246
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id            0
hadm_id               0
stay_id               0
charttime             0
itemid                0
valuenum      263786245
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id       stay_id        itemid      valuenum
count  4.329975e+08  4.329975e+08  4.329975e+08  4.329975e+08  1.692112e+08
mean   1.501872e+07  2.500668e+07  3.497639e+07  2.250174e+05  8.232441e+01
std    2.892695e+06  2.871356e+06  2.886502e+06  2.921064e+03  1.699129e+04
min    1.000003e+07  2.000009e+07  3.000015e+07  2.

In [13]:
labevents.head()

,subject_id,hadm_id,itemid,charttime,valuenum
0,10000032,NaN,50931,2180-03-23 11:51:00,95.0
1,10000032,NaN,51071,2180-03-23 11:51:00,NaN
2,10000032,NaN,51074,2180-03-23 11:51:00,NaN
3,10000032,NaN,51075,2180-03-23 11:51:00,NaN
4,10000032,NaN,51079,2180-03-23 11:51:00,NaN


In [ ]:
print(f'''labevents:
{labevents.shape}
{'-'*100}
{labevents.dtypes}
{'-'*100}
{labevents.count()}
{'-'*100}
{labevents.isna().sum()}
{'-'*100}
{labevents.describe()}
{'-'*100}
''')

labevents:
(158374764, 5)
----------------------------------------------------------------------------------------------------
subject_id    158374764
hadm_id        84605867
itemid        158374764
charttime     158374764
valuenum      136884423
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id           0
hadm_id       73768897
itemid               0
charttime            0
valuenum      21490341
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id        itemid      valuenum
count  1.583748e+08  8.460587e+07  1.583748e+08  1.368844e+08
mean   1.501474e+07  2.500166e+07  5.117165e+04  6.882403e+01
std    2.883378e+06  2.883109e+06  3.182457e+02  2.885397e+03
min    1.000003e+07  2.000002e+07  5.080100e+04 -1.444000e+03
25%    1.251950e+07  2.250096e+07  5.092400e+04  3.700000e+00
50%    1.501658e+07  2.501462e+07  5.1

In [14]:
d_items.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [ ]:
print(f'''d_items:
{d_items.shape}
{'-'*100}
{d_items.dtypes}
{'-'*100}
{d_items.count()}
{'-'*100}
{d_items.isna().sum()}
{'-'*100}
{d_items.describe()}
{'-'*100}
''')

d_items:
(4095, 9)
----------------------------------------------------------------------------------------------------
itemid             4095
label              4095
abbreviation       4095
linksto            4095
category           4095
unitname           1123
param_type         4095
lownormalvalue       19
highnormalvalue      22
dtype: int64
----------------------------------------------------------------------------------------------------
itemid                0
label                 0
abbreviation          0
linksto               0
category              0
unitname           2972
param_type            0
lownormalvalue     4076
highnormalvalue    4073
dtype: int64
----------------------------------------------------------------------------------------------------
              itemid  lownormalvalue  highnormalvalue
count    4095.000000       19.000000        22.000000
mean   226607.692796       65.000000       133.640909
std      2493.494899      107.356623       253.076007
min 

In [17]:
d_labitems.head()

,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


In [ ]:
print(f'''d_labitems:
{d_labitems.shape}
{'-'*100}
{d_labitems.dtypes}
{'-'*100}
{d_labitems.count()}
{'-'*100}
{d_labitems.isna().sum()}
{'-'*100}
{d_labitems.describe()}
{'-'*100}
''')

d_labitems:
(1650, 4)
----------------------------------------------------------------------------------------------------
itemid      1650
label       1646
fluid       1650
category    1650
dtype: int64
----------------------------------------------------------------------------------------------------
itemid      0
label       4
fluid       0
category    0
dtype: int64
----------------------------------------------------------------------------------------------------
             itemid
count   1650.000000
mean   51734.312727
std      602.758734
min    50801.000000
25%    51227.250000
50%    51705.500000
75%    52147.750000
max    53190.000000
----------------------------------------------------------------------------------------------------

